# 定义网址

In [ ]:
url="https://www.ricedata.cn/variety/identified/nation_1.htm"

# 发送请求

In [ ]:
import requests

In [ ]:
res=requests.get(url)

In [ ]:
res.status_code

# 解析数据

In [ ]:
res.encoding='GB2312'
#用‘GB2312’转码

In [ ]:
res.encoding

In [ ]:
res.text

## 品种名称

In [ ]:
#用xpath方法抓取
from lxml import etree 
html=etree.HTML(res.text)

In [ ]:
variety=html.xpath('//a[@target="_blank"]/text()')
variety

## 亲本来源（"×"前为母本）

In [ ]:
temp=html.xpath('//td[@style="border-top:dotted silver 1px"]/text()')
temp

In [ ]:
#提出第2,7,12等项为亲本来源
source_of_parent=temp[1::5]
source_of_parent

## 类型

In [ ]:
#提出第3,8,13等项为类型
type_=temp[2::5]
type_

## 原产地/选育单位

In [ ]:
#提出第4,9,14等项为原产地
origin=temp[3::5]
origin

## 审定编号

In [ ]:
#提出第5,10,15等项为审定编号
number=temp[4::5]
number

# 实现翻页抓取

In [ ]:
#找到规律：https://www.ricedata.cn/variety/identified/nation_[PAGE].htm
#中间的PAGE为第几页，发现最大页数为99页
max_page=99
varietys=[]
source_of_parents=[]
type_s=[]
origins=[]
numbers=[]


for page in range(1,max_page+1):
    url=f'https://www.ricedata.cn/variety/identified/nation_{page}.htm'
    res=requests.get(url)
    res.encoding='GB2312'
    html=etree.HTML(res.text)
    variety=html.xpath('//a[@target="_blank"]/text()')
    temp=html.xpath('//td[@style="border-top:dotted silver 1px"]/text()')
    source_of_parent=temp[1::5]
    type_=temp[2::5]
    origin=temp[3::5]
    number=temp[4::5]
    
    varietys.extend(variety)
    source_of_parents.extend(source_of_parent)
    type_s.extend(type_)
    origins.extend(origin)
    numbers.extend(number)
    
    print(f'{page}/{max_page}-------{len(variety),len(source_of_parent),len(type_),len(origin),len(number)}')


# 数据储存

In [ ]:
import pandas as pd

In [ ]:
data=pd.DataFrame([varietys,source_of_parents,type_s,origins,numbers],
                  index=[' 品种名称','亲本来源（"×"前为母本）','类型',
                         '原产地/选育单位','审定编号']).T
data

In [ ]:
#存储为Excel文件
data.to_excel('农产品数据采集与分析.xlsx')

# 数据分析

## 计算不同类型水稻的数量，并绘制出饼状图

### 计算数量

In [ ]:
type_s_dict={}
for i in type_s:
    if  i not in type_s_dict:
        type_s_dict[i]=0
    type_s_dict[i]+=1
print(type_s_dict)

### 绘制饼状图

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 更换默认字体，使得中文显示正常
plt.rcParams['axes.unicode_minus'] = False  # 解决更换字体后导致的负号显示乱码

In [ ]:
value=list(type_s_dict.values()),
key=list(type_s_dict.keys())

In [ ]:
values=[]
for i in value:
    values.extend(i)

In [ ]:
values

In [ ]:
plt.figure(figsize=(650,13), dpi=100)
# x：表示用来绘制饼状图的数据
# labels：表示每个数据对应的标签
# explode：表示每块扇形区域离中心点的距离
# autopct：表示显示每块扇形区域的格式
# colors：每块扇形区域的颜色
plt.pie(x=values,
        labels=key, 
        explode=[0]*7,
        autopct='%.1f%%',
        colors=['#F7ADBC', '#DEAB8A', '#847E39', '#444793', '#B2D235', '#F05B72', '#2E3920', '#FAB37B', '#87481F'])
plt.show()

## 计算不同选育单位选育水稻的数量，并绘制出柱状图。

### 计算不同选育单位选育水稻的数量

In [ ]:
import jieba

In [ ]:
from collections import Counter

In [ ]:
count_word = Counter(res)
dict(count_word)

In [ ]:
count={}
for i in count_word.keys():
    tp=i
    if  '等' in i:    
        tp=i.replace('等','')
    if tp not in count:
        count[tp]=1
    count[tp]+=count_word[i]
    print(count) 
        
        

### 绘制出柱状图

In [ ]:
plt.figure(figsize=(100, 100), dpi=100)
# x：横坐标标签
# height：每根柱子的高度
# width：柱子的宽度，默认情况下等于0.8
# color：每根柱子的颜色
plt.bar(x=count_word.keys(), 
        height=count_word.values(), 
        width=0.5, 
        color=['#F7ADBC', '#DEAB8A', '#847E39', '#444793', '#B2D235', '#F05B72', '#2E3920', '#FAB37B', '#87481F'])
plt.show()